In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("postgres_to_parquet_silver") \
    .master("spark://spark-master:7077") \
    .config("spark.default.parallelism", "4") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.driver.memory", "1g") \
    .config("spark.sql.autoBroadcastJoinThreshold", "10m") \
    .config("spark.sql.parquet.enableVectorizedReader", "true") \
    .config("spark.sql.legacy.parquet.nanosAsLong", "true") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minio") \
    .config("spark.hadoop.fs.s3a.secret.key", "minio123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/28 22:43:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
# ============================================================
# 1. LOAD SILVER DATASETS ONCE
# ============================================================

df_users_silver = spark.read.parquet("s3a://end-to-end-streaming-data-platform-silver/postgres/users.parquet")
df_subscriptions_silver = spark.read.parquet("s3a://end-to-end-streaming-data-platform-silver/postgres/subscription.parquet")

df_videos_silver = spark.read.parquet("s3a://end-to-end-streaming-data-platform-silver/mongo/videos_silver.parquet")
df_events_silver = spark.read.parquet("s3a://end-to-end-streaming-data-platform-silver/kafka/events_silver.parquet")


26/09/28 22:44:15 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

In [3]:
# 1. Event + User
context = (
    df_events_silver.alias("e")
    .join(
        df_users_silver.alias("u"),
        F.col("e.user_id") == F.col("u.user_id"),
        "inner",
    )
    # 2. + Video
    .join(
        df_videos_silver.alias("v"),
        F.col("e.video_id") == F.col("v.video_id"),
        "inner",
    )
    # 3. + Subscription التي تغطي وقت الـ Event
    .join(
        df_subscriptions_silver.alias("s"),
        (
            (F.col("e.user_id") == F.col("s.user_id"))
            & (F.col("e.event_timestamp") >= F.col("s.start_date"))
            & (F.col("e.event_timestamp") <= F.col("s.end_date"))
        ),
        "inner",
    )
)

In [4]:
unified_context = context.select(
    # Event
    F.col("e.event_id"),
    F.col("e.user_id"),
    F.col("e.video_id"),
    F.col("e.event_timestamp"),
    F.col("e.interaction_type"),
    F.col("e.watch_time_sec"),
    F.col("e.device_type"),

    # User
    F.col("u.created_at").alias("user_created_at"),
    F.col("u.country"),
    F.col("u.account_status"),

    # Subscription
    F.col("s.subscription_id"),
    F.col("s.plan_type"),
    F.col("s.start_date").alias("subscription_start_date"),
    F.col("s.end_date").alias("subscription_end_date"),
    F.col("s.is_active"),
    F.col("s.auto_renew"),

    # Video
    F.col("v.created_at").alias("video_created_at"),
    F.col("v.duration_sec").alias("video_duration_sec"),
    F.col("v.content_type").alias("video_content_type"),
    F.col("v.status").alias("video_status"),
)

In [5]:
df_payments_silver = spark.read.parquet("s3a://end-to-end-streaming-data-platform-silver/postgres/payments.parquet")

In [6]:
payment_context = (
    df_payments_silver
    .groupBy("subscription_id")
    .agg(
        F.count("*").alias("payment_attempt_count"),

        F.sum(
            F.when(
                F.col("payment_status") == "success",
                1
            ).otherwise(0)
        ).alias("successful_payment_count"),

        F.sum(
            F.when(
                F.col("payment_status") == "failed",
                1
            ).otherwise(0)
        ).alias("failed_payment_count"),

        F.sum(
            F.when(
                F.col("payment_status") == "success",
                F.col("amount")
            ).otherwise(F.lit(0))
        ).alias("successful_payment_total"),

        F.min(
            F.when(
                F.col("payment_status") == "success",
                F.col("amount")
            )
        ).alias("successful_payment_min_amount"),

        F.max(
            F.when(
                F.col("payment_status") == "success",
                F.col("amount")
            )
        ).alias("successful_payment_max_amount"),
    )
)

In [7]:
payment_context = (
    payment_context
    .withColumn(
        "has_successful_payment",
        F.col("successful_payment_count") > 0
    )
    .withColumn(
        "has_failed_payment",
        F.col("failed_payment_count") > 0
    )
    .withColumn(
        "payment_amount_consistent",
        F.col("successful_payment_min_amount")
        == F.col("successful_payment_max_amount")
    )
)

In [8]:
unified_context = (
    unified_context.alias("uc")
    .join(
        payment_context.alias("pc"),
        F.col("uc.subscription_id") == F.col("pc.subscription_id"),
        "left",
    )
)

In [9]:
unified_context = unified_context.select(
    F.col("uc.*"),

    F.col("pc.payment_attempt_count"),
    F.col("pc.successful_payment_count"),
    F.col("pc.failed_payment_count"),
    F.col("pc.successful_payment_total"),
    F.col("pc.successful_payment_min_amount"),
    F.col("pc.successful_payment_max_amount"),
    F.col("pc.has_successful_payment"),
    F.col("pc.has_failed_payment"),
    F.col("pc.payment_amount_consistent"),
)

In [ ]:
# 4.Apply Structural Rules

In [10]:
unified_context = (
    unified_context

    # Required identifiers
    .withColumn(
        "is_missing_event_id",
        F.col("event_id").isNull()
    )
    .withColumn(
        "is_missing_user_id",
        F.col("user_id").isNull()
    )
    .withColumn(
        "is_missing_video_id",
        F.col("video_id").isNull()
    )
    .withColumn(
        "is_missing_subscription_id",
        F.col("subscription_id").isNull()
    )

    # Event attributes
    .withColumn(
        "is_missing_event_timestamp",
        F.col("event_timestamp").isNull()
    )
    .withColumn(
        "is_missing_interaction_type",
        F.col("interaction_type").isNull()
    )

    # Context attributes
    .withColumn(
        "is_missing_user_created_at",
        F.col("user_created_at").isNull()
    )
    .withColumn(
        "is_missing_video_created_at",
        F.col("video_created_at").isNull()
    )
)

In [11]:
unified_context = (
    unified_context
    .withColumn(
        "structural_valid",
        ~(
            F.col("is_missing_event_id")
            | F.col("is_missing_user_id")
            | F.col("is_missing_video_id")
            | F.col("is_missing_subscription_id")
            | F.col("is_missing_event_timestamp")
            | F.col("is_missing_interaction_type")
            | F.col("is_missing_user_created_at")
            | F.col("is_missing_video_created_at")
        )
    )
)

In [12]:
duplicate_events = (
    unified_context
    .groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
)

In [13]:
duplicate_event_count = duplicate_events.count()

print("Duplicate event IDs:", duplicate_event_count)

[Stage 8:>                                                          (0 + 1) / 1]

Duplicate event IDs: 10758


In [14]:
event_subscription_matches = (
    df_events_silver.alias("e")
    .join(
        df_subscriptions_silver.alias("s"),
        (
            (F.col("e.user_id") == F.col("s.user_id"))
            & (F.col("e.event_timestamp") >= F.col("s.start_date"))
            & (F.col("e.event_timestamp") <= F.col("s.end_date"))
        ),
        "left",
    )
    .groupBy("e.event_id")
    .agg(
        F.count("s.subscription_id").alias("subscription_match_count")
    )
)

In [15]:
event_subscription_matches.groupBy(
    "subscription_match_count"
).count().orderBy(
    "subscription_match_count"
).show()

+------------------------+-----+
|subscription_match_count|count|
+------------------------+-----+
|                       0|    1|
|                       1| 4921|
|                       2| 5353|
|                       3| 5405|
+------------------------+-----+



In [16]:
from pyspark.sql import Window


subscription_window = (
    Window
    .partitionBy("event_id")
    .orderBy(
        F.col("subscription_start_date").desc(),
        F.col("subscription_id").desc(),
    )
)

unified_context = (
    unified_context
    .withColumn(
        "subscription_rank",
        F.row_number().over(subscription_window)
    )
    .filter(F.col("subscription_rank") == 1)
    .drop("subscription_rank")
)

In [17]:
duplicate_events = (
    unified_context
    .groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_event_count = duplicate_events.count()

print("Duplicate event IDs:", duplicate_event_count)

[Stage 24:>                                                         (0 + 1) / 1]

Duplicate event IDs: 0


In [18]:
 # 1. User relationship
orphan_users = (
    unified_context
    .join(
        df_users_silver.select("user_id").distinct(),
        "user_id",
        "left_anti",
    )
    .count()
)

# 2. Video relationship
orphan_videos = (
    unified_context
    .join(
        df_videos_silver.select("video_id").distinct(),
        "video_id",
        "left_anti",
    )
    .count()
)

# 3. Subscription relationship
orphan_subscriptions = (
    unified_context
    .join(
        df_subscriptions_silver.select("subscription_id").distinct(),
        "subscription_id",
        "left_anti",
    )
    .count()
)

print("Orphan user IDs:", orphan_users)
print("Orphan video IDs:", orphan_videos)
print("Orphan subscription IDs:", orphan_subscriptions)

Orphan user IDs: 0
Orphan video IDs: 0
Orphan subscription IDs: 0


In [19]:
unified_context = (
    unified_context
    .withColumn(
        "is_event_before_user",
        F.col("event_timestamp") < F.col("user_created_at")
    )
    .withColumn(
        "is_event_before_video",
        F.col("event_timestamp") < F.col("video_created_at")
    )
)

In [20]:
event_before_user = (
    unified_context
    .filter(F.col("is_event_before_user"))
    .count()
)

event_before_video = (
    unified_context
    .filter(F.col("is_event_before_video"))
    .count()
)

print("Events before user creation:", event_before_user)
print("Events before video creation:", event_before_video)

Events before user creation: 0
Events before video creation: 0


In [ ]:
unified_context = (
    unified_context
    .withColumn(
        "is_event_before_user",
        F.col("event_timestamp") < F.col("user_created_at")
    )
    .withColumn(
        "is_event_before_video",
        F.col("event_timestamp") < F.col("video_created_at")
    )
)

print(
    "Events before user creation:",
    unified_context.filter(F.col("is_event_before_user")).count()
)

print(
    "Events before video creation:",
    unified_context.filter(F.col("is_event_before_video")).count()
)

In [21]:
unified_context = (
    unified_context
    .withColumn(
        "is_event_outside_subscription",
        (
            (F.col("event_timestamp") < F.col("subscription_start_date"))
            | (F.col("event_timestamp") > F.col("subscription_end_date"))
        )
    )
)

event_outside_subscription = (
    unified_context
    .filter(F.col("is_event_outside_subscription"))
    .count()
)

print(
    "Events outside subscription period:",
    event_outside_subscription
)

Events outside subscription period: 0


In [22]:
unified_context = (
    unified_context
    .withColumn(
        "is_subscription_before_user",
        F.col("subscription_start_date") < F.to_date(F.col("user_created_at"))
    )
)

subscription_before_user = (
    unified_context
    .filter(F.col("is_subscription_before_user"))
    .count()
)

print(
    "Subscriptions before user creation:",
    subscription_before_user
)

Subscriptions before user creation: 0


In [23]:
unified_context = (
    unified_context
    .withColumn(
        "is_invalid_subscription_period",
        F.col("subscription_start_date") > F.col("subscription_end_date")
    )
)

invalid_subscription_period = (
    unified_context
    .filter(F.col("is_invalid_subscription_period"))
    .count()
)

print(
    "Invalid subscription periods:",
    invalid_subscription_period
)

Invalid subscription periods: 0


In [24]:
unified_context = (
    unified_context
    .withColumn(
        "temporal_valid",
        ~(
            F.col("is_event_before_user")
            | F.col("is_event_before_video")
            | F.col("is_event_outside_subscription")
            | F.col("is_subscription_before_user")
            | F.col("is_invalid_subscription_period")
        )
    )
)

print(
    "Temporal invalid rows:",
    unified_context.filter(~F.col("temporal_valid")).count()
)

Temporal invalid rows: 0


In [25]:
unified_context = (
    unified_context
    .withColumn(
        "is_negative_watch_time",
        F.col("watch_time_sec") < 0
    )
)

negative_watch_time = (
    unified_context
    .filter(F.col("is_negative_watch_time"))
    .count()
)

print(
    "Negative watch time:",
    negative_watch_time
)

[Stage 123:>                                                        (0 + 1) / 1]

Negative watch time: 0


In [26]:
unified_context = (
    unified_context
    .withColumn(
        "is_watch_time_exceeds_duration",
        (
            F.col("watch_time_sec").isNotNull()
            & F.col("video_duration_sec").isNotNull()
            & (F.col("video_duration_sec") > 0)
            & (F.col("watch_time_sec") > F.col("video_duration_sec"))
        )
    )
)

watch_time_exceeds_duration = (
    unified_context
    .filter(F.col("is_watch_time_exceeds_duration"))
    .count()
)

print(
    "Watch time exceeds video duration:",
    watch_time_exceeds_duration
)

Watch time exceeds video duration: 0


In [27]:
unified_context = (
    unified_context
    .withColumn(
        "is_invalid_interaction_type",
        ~F.col("interaction_type").isin(
            "play",
            "complete",
            "like",
            "pause",
        )
    )
)

invalid_interaction_type = (
    unified_context
    .filter(F.col("is_invalid_interaction_type"))
    .count()
)

print(
    "Invalid interaction types:",
    invalid_interaction_type
)

Invalid interaction types: 1


In [28]:
invalid_interaction_type.s

AttributeError: 'int' object has no attribute 'show'

In [29]:
unified_context \
    .filter(F.col("is_invalid_interaction_type")) \
    .select(
        "event_id",
        "user_id",
        "video_id",
        "interaction_type"
    ) \
    .show(truncate=False)

+------------------------------------+------------------------------------+------------------------------------+----------------+
|event_id                            |user_id                             |video_id                            |interaction_type|
+------------------------------------+------------------------------------+------------------------------------+----------------+
|01a076ee-6c45-7748-b089-510cda3bd837|01a076ee-6c41-728d-95c9-e39cb4a6b11c|01a076ee-4ffc-74ff-a988-06d0e8473a45|share           |
+------------------------------------+------------------------------------+------------------------------------+----------------+



In [30]:
unified_context = (
    unified_context
    .withColumn(
        "is_invalid_device_type",
        ~F.col("device_type").isin(
            "mobile",
            "tv",
            "web",
        )
    )
)

invalid_device_type = (
    unified_context
    .filter(F.col("is_invalid_device_type"))
    .count()
)

print(
    "Invalid device types:",
    invalid_device_type
)

Invalid device types: 1


In [31]:
unified_context \
    .filter(F.col("is_invalid_device_type")) \
    .select(
        "event_id",
        "user_id",
        "video_id",
        "device_type"
    ) \
    .show(truncate=False)

+------------------------------------+------------------------------------+------------------------------------+-----------+
|event_id                            |user_id                             |video_id                            |device_type|
+------------------------------------+------------------------------------+------------------------------------+-----------+
|01a076ee-6c46-7b1c-8ba4-facb6afc6d30|01a076ee-6c41-728d-95c9-e39cb4a6b11c|01a076ee-537d-764c-8a16-5932f3806d03|console    |
+------------------------------------+------------------------------------+------------------------------------+-----------+



In [32]:
unified_context = (
    unified_context
    .withColumn(
        "is_invalid_plan_type",
        ~F.col("plan_type").isin(
            "free",
            "premium",
        )
    )
)

invalid_plan_type = (
    unified_context
    .filter(F.col("is_invalid_plan_type"))
    .count()
)

print(
    "Invalid plan types:",
    invalid_plan_type
)

Invalid plan types: 0


In [33]:
unified_context = (
    unified_context
    .withColumn(
        "is_payment_amount_inconsistent",
        (
            F.col("has_successful_payment")
            & ~F.col("payment_amount_consistent")
        )
    )
)

payment_amount_inconsistent = (
    unified_context
    .filter(F.col("is_payment_amount_inconsistent"))
    .count()
)

print(
    "Subscriptions with inconsistent successful payment amounts:",
    payment_amount_inconsistent
)

Subscriptions with inconsistent successful payment amounts: 0


In [34]:
unified_context = (
    unified_context
    .withColumn(
        "is_free_with_successful_payment",
        (
            (F.col("plan_type") == "free")
            & F.col("has_successful_payment")
        )
    )
)

free_with_successful_payment = (
    unified_context
    .filter(F.col("is_free_with_successful_payment"))
    .count()
)

print(
    "Free subscriptions with successful payments:",
    free_with_successful_payment
)

Free subscriptions with successful payments: 0


In [35]:
unified_context = (
    unified_context
    .withColumn(
        "is_premium_without_successful_payment",
        (
            (F.col("plan_type") == "premium")
            & ~F.col("has_successful_payment")
        )
    )
)

premium_without_successful_payment = (
    unified_context
    .filter(F.col("is_premium_without_successful_payment"))
    .count()
)

print(
    "Premium subscriptions without successful payments:",
    premium_without_successful_payment
)

Premium subscriptions without successful payments: 27


In [37]:
unified_context \
    .filter(
        (F.col("plan_type") == "premium")
        & ~F.col("has_successful_payment")
    ) \
    .select(
        "subscription_id",
        "payment_attempt_count",
        "successful_payment_count",
        "failed_payment_count",
        "has_successful_payment",
        "has_failed_payment",
    ) \
    .dropDuplicates() \
    .groupBy(
        "payment_attempt_count",
        "successful_payment_count",
        "failed_payment_count",
        "has_successful_payment",
        "has_failed_payment",
    ) \
    .count() \
    .orderBy("payment_attempt_count") \
    .show(truncate=False)

+---------------------+------------------------+--------------------+----------------------+------------------+-----+
|payment_attempt_count|successful_payment_count|failed_payment_count|has_successful_payment|has_failed_payment|count|
+---------------------+------------------------+--------------------+----------------------+------------------+-----+
|3                    |0                       |3                   |false                 |true              |9    |
+---------------------+------------------------+--------------------+----------------------+------------------+-----+



In [38]:
invalid_payment_status = (
    df_payments_silver
    .filter(
        ~F.col("payment_status").isin(
            "success",
            "failed",
        )
    )
    .count()
)

print("Invalid payment statuses:", invalid_payment_status)

Invalid payment statuses: 0


In [40]:
df_payments_silver.printSchema()

root
 |-- payment_id: string (nullable = true)
 |-- subscription_id: string (nullable = true)
 |-- amount: decimal(10,2) (nullable = true)
 |-- currency: string (nullable = true)
 |-- payment_date: timestamp (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- payment_date_day: date (nullable = true)



In [41]:
df_subscriptions_silver.printSchema()

root
 |-- subscription_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- plan_type: string (nullable = true)
 |-- start_date: timestamp (nullable = true)
 |-- end_date: timestamp (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- auto_renew: boolean (nullable = true)
 |-- is_end_date_before_start_date: boolean (nullable = true)
 |-- start_date_day: date (nullable = true)
 |-- end_date_day: date (nullable = true)



In [47]:
payment_temporal_check = (
    df_payments_silver.alias("p")
    .join(
        df_subscriptions_silver.select(
            "subscription_id",
            "start_date",
            "end_date"
        ).alias("s"),
        F.col("p.subscription_id") == F.col("s.subscription_id"),
        "inner"
    )
    .filter(
        F.col("p.payment_date")
        < F.col("s.start_date")
    )
)

payments_before_subscription = payment_temporal_check.count()

print(
    "Payments before subscription start:",
    payments_before_subscription
)

Payments before subscription start: 0


In [54]:
payment_after_subscription = (
    df_payments_silver.alias("p")
    .join(
        df_subscriptions_silver.select(
            "subscription_id",
            "end_date"
        ).alias("s"),
        F.col("p.subscription_id") == F.col("s.subscription_id"),
        "inner"
    )
    .filter(
        F.col("p.payment_date")
        > F.col("s.end_date")
    )
    .show(truncate=False,vertical=True)
)

print(
    "Payments after subscription end:",
    payment_after_subscription
)

-RECORD 0------------------------------------------------
 payment_id       | 01a076ee-6c3a-7df0-975c-dc0760eaae99 
 subscription_id  | 01a076ee-6c39-7206-b67b-344f353021fb 
 amount           | 14.99                                
 currency         | USD                                  
 payment_date     | 2026-01-01 00:00:00                  
 payment_status   | success                              
 payment_date_day | 2026-01-01                           
 subscription_id  | 01a076ee-6c39-7206-b67b-344f353021fb 
 end_date         | 2025-12-31 00:00:00                  

Payments after subscription end: None


In [52]:
payment_temporal_check \
    .filter(
        F.col("p.payment_date")
        > F.col("s.end_date")
    ) \
    .select(
        F.col("p.payment_id"),
        F.col("p.subscription_id"),
        F.col("p.payment_date"),
        F.col("s.end_date"),
        F.col("p.payment_status"),
    ) \
    .count()

0

In [55]:
payment_business_context = (
    df_payments_silver
    .groupBy("subscription_id")
    .agg(
        F.max(
            F.when(
                ~F.col("payment_status").isin("success", "failed"),
                1
            ).otherwise(0)
        ).alias("has_invalid_payment_status")
    )
)

In [57]:
payment_after_end_context = (
    payment_temporal_check
    .select("p.subscription_id")
    .distinct()
    .withColumn("has_payment_after_end", F.lit(1))
)

In [58]:
unified_context = (
    unified_context
    .join(
        payment_business_context,
        "subscription_id",
        "left"
    )
    .join(
        payment_after_end_context,
        "subscription_id",
        "left"
    )
    .fillna({
        "has_invalid_payment_status": 0,
        "has_payment_after_end": 0,
    })
)

In [59]:
unified_context = (
    unified_context
    .withColumn(
        "business_valid",
        ~(
            F.col("is_negative_watch_time")
            | F.col("is_watch_time_exceeds_duration")
            | F.col("is_invalid_interaction_type")
            | F.col("is_invalid_device_type")
            | F.col("is_invalid_plan_type")
            | F.col("is_payment_amount_inconsistent")
            | F.col("is_free_with_successful_payment")
            | (F.col("has_invalid_payment_status") == 1)
            | (F.col("has_payment_after_end") == 1)
        )
    )
)

print(
    "Business invalid rows:",
    unified_context.filter(~F.col("business_valid")).count()
)

[Stage 275:>                                                        (0 + 1) / 1]

Business invalid rows: 2


In [60]:
unified_context = (
    unified_context
    .withColumn(
        "gate_valid",
        (
            F.col("structural_valid")
            & F.col("temporal_valid")
            & F.col("business_valid")
        )
    )
)

print(
    "Final invalid context rows:",
    unified_context.filter(~F.col("gate_valid")).count()
)

print(
    "Final valid context rows:",
    unified_context.filter(F.col("gate_valid")).count()
)

Final invalid context rows: 2
Final valid context rows: 2583


In [61]:
valid_context = (
    unified_context
    .filter(F.col("gate_valid"))
)

In [62]:
duplicate_valid_events = (
    valid_context
    .groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate valid event IDs:",
    duplicate_valid_events.count()
)

Duplicate valid event IDs: 0


In [ ]:
#7

In [63]:
valid_context.printSchema()

root
 |-- subscription_id: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- video_id: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- interaction_type: string (nullable = true)
 |-- watch_time_sec: integer (nullable = true)
 |-- device_type: string (nullable = true)
 |-- user_created_at: timestamp (nullable = true)
 |-- country: string (nullable = true)
 |-- account_status: string (nullable = true)
 |-- plan_type: string (nullable = true)
 |-- subscription_start_date: timestamp (nullable = true)
 |-- subscription_end_date: timestamp (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- auto_renew: boolean (nullable = true)
 |-- video_created_at: timestamp (nullable = true)
 |-- video_duration_sec: integer (nullable = true)
 |-- video_content_type: string (nullable = true)
 |-- video_status: string (nullable = true)
 |-- payment_attempt_count: long (nullable = true)
 |-- successful_pay